# Modello con MobilNet

- Descrizione delle cose che abbiamo fatto in questo Notebook-

## Import

In [1]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision.models as models
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import numpy as np
from sklearn.metrics import balanced_accuracy_score,confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns


## Caricamento Dataset Split

In [ ]:
OUTPUT_CSV = '../train_val_split.csv'

#carichiamo il dataset
try:
    df = pd.read_csv(OUTPUT_CSV)
    train_df = df[df['split'] == 'train']
    val_df = df[df['split'] == 'val']
    print(f" CSV caricato: {len(train_df)} immagini di Train, {len(val_df)} di Validation.")
except FileNotFoundError:
    raise FileNotFoundError(f" ERRORE: Non trovo il file {OUTPUT_CSV}.")

#data loader
print("\n--- FASE 2: Preparazione Tensori ---")

base_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

class WasteDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe
        self.transform = transform

    def __len__(self): return len(self.dataframe)

    def __getitem__(self, idx):
        img_path = self.dataframe.iloc[idx]['filepath']
        image = Image.open(img_path).convert('RGB')
        label = self.dataframe.iloc[idx]['label']
        if self.transform: image = self.transform(image)
        return image, label

BATCH_SIZE = 32 
train_loader = DataLoader(WasteDataset(train_df, transform=base_transforms), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(WasteDataset(val_df, transform=base_transforms), batch_size=BATCH_SIZE, shuffle=False)
print(" Dataloader pronti!")

## Caricamento MobilNet, adeguamento e addestramento


### SGD

In [ ]:

#TRANSFER LEARNING (MobileNetV2)
print("\n--- FASE 3: Inizializzazione MobileNetV2 ---")


weights = models.MobileNet_V2_Weights.DEFAULT
model = models.mobilenet_v2(weights=weights)


for param in model.parameters():
    param.requires_grad = False


num_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(num_features, 8)

device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)


criterion = nn.CrossEntropyLoss()
# NUOVA RIGA (SGD con Momentum)
# Nota: L'SGD di solito ha bisogno di un Learning Rate di partenza leggermente più alto (0.01)
optimizer = optim.SGD(model.classifier[1].parameters(), lr=0.01, momentum=0.9, weight_decay=1e-4)

print(f" MobileNetV2 pronta. Parametri congelati, testa modificata per 8 classi.")
print(f" Acceleratore in uso: {device.type.upper()}")

#addestramento e accuracy
print("\n--- FASE 4: Addestramento ---")

EPOCHS = 10

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    for i, (inputs, labels) in enumerate(train_loader):
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        if i % 20 == 19: 
            print(f"  Epoca [{epoch+1}/{EPOCHS}], Batch [{i+1}/{len(train_loader)}], Loss: {running_loss/20:.4f}")
            running_loss = 0.0
            
    print(f" Epoca {epoch+1} conclusa. Calcolo metriche di validazione...")
    
    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs = inputs.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.numpy())
            
    # Calcolo della Balanced Accuracy richiesta dal progetto
    bal_acc = balanced_accuracy_score(all_labels, all_preds) * 100
    print(f"🏆 Balanced Accuracy all'epoca {epoch+1}: {bal_acc:.2f}%\n")

torch.save(model.state_dict(), 'mobilenet_mivia_weights_3.pth')
print("🎉 Addestramento concluso! Modello salvato in: mobilenet_mivia_weights.pth")

🚀 MIVIA DAY 3: Transfer Learning con MobileNetV2 🚀

 CSV caricato: 12412 immagini di Train, 3103 di Validation.

--- FASE 2: Preparazione Tensori ---
 Dataloader pronti!

--- FASE 3: Inizializzazione MobileNetV2 ---
 MobileNetV2 pronta. Parametri congelati, testa modificata per 8 classi.
 Acceleratore in uso: MPS

--- FASE 4: Addestramento ---
  Epoca [1/10], Batch [20/388], Loss: 1.4009
  Epoca [1/10], Batch [40/388], Loss: 0.8046
  Epoca [1/10], Batch [60/388], Loss: 0.5700
  Epoca [1/10], Batch [80/388], Loss: 0.4825
  Epoca [1/10], Batch [100/388], Loss: 0.4301
  Epoca [1/10], Batch [120/388], Loss: 0.3774
  Epoca [1/10], Batch [140/388], Loss: 0.4189
  Epoca [1/10], Batch [160/388], Loss: 0.3143
  Epoca [1/10], Batch [180/388], Loss: 0.3094
  Epoca [1/10], Batch [200/388], Loss: 0.3733
  Epoca [1/10], Batch [220/388], Loss: 0.3084
  Epoca [1/10], Batch [240/388], Loss: 0.3259
  Epoca [1/10], Batch [260/388], Loss: 0.2909
  Epoca [1/10], Batch [280/388], Loss: 0.2988
  Epoca [1/10]

### Adam

In [ ]:
#TRANSFER LEARNING (MobileNetV2)
print("\n--- FASE 3: Inizializzazione MobileNetV2 ---")


weights = models.MobileNet_V2_Weights.DEFAULT
model = models.mobilenet_v2(weights=weights)


for param in model.parameters():
    param.requires_grad = False


num_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(num_features, 8)

device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)


criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.classifier[1].parameters(), lr=0.001)

print(f" MobileNetV2 pronta. Parametri congelati, testa modificata per 8 classi.")
print(f" Acceleratore in uso: {device.type.upper()}")

#addestramento e accuracy
print("\n--- FASE 4: Addestramento ---")

EPOCHS = 5 

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    for i, (inputs, labels) in enumerate(train_loader):
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        if i % 20 == 19: 
            print(f"  Epoca [{epoch+1}/{EPOCHS}], Batch [{i+1}/{len(train_loader)}], Loss: {running_loss/20:.4f}")
            running_loss = 0.0
            
    print(f" Epoca {epoch+1} conclusa. Calcolo metriche di validazione...")
    
    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs = inputs.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.numpy())
            
    # Calcolo della Balanced Accuracy richiesta dal progetto
    bal_acc = balanced_accuracy_score(all_labels, all_preds) * 100
    print(f"🏆 Balanced Accuracy all'epoca {epoch+1}: {bal_acc:.2f}%\n")

torch.save(model.state_dict(), 'mobilenet_mivia_weights.pth')
print("🎉 Addestramento concluso! Modello salvato in: mobilenet_mivia_weights.pth")

In [5]:
!pip install matplotlib seaborn

  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
  Using cached contourpy-1.3.3-cp314-cp314-macosx_11_0_arm64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached pyparsing-3.3.2-py3-none-any.whl.metadata (5.8 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.3/9.3 MB 8.9 MB/s  0:00:01m0:00:0100:01
Using cached seaborn-0.13.2-py3-none-any.whl (294 kB)
Using cached contourpy-1.3.3-cp314-cp314-macosx_11_0_arm64.whl (273 kB)
Using cached cycler-0.12.1-py3-none-any.whl (8.3 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 9.5 MB/s  0:00:00 eta 0:00:01
Using cached pyparsing-3.3.2-py3-none-any.whl (122 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [seaborn]m6/7 [seaborn]ib]

[notice] A new release of pip is available: 26.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


## Analisi Risultati

In [1]:


# Nomi delle classi per i grafici
class_names = ['Battery', 'Clothing', 'Glass', 'Metal', 'Organic', 'Papery', 'Plastic', 'Undiff']
model.eval()  

# ==========================================
# 1. VALUTAZIONE SUL TRAINING SET
# ==========================================
print("🏋️ Analisi del Training Set in corso...")
# Creiamo un loader per il training senza mischiarlo (shuffle=False) per una valutazione pulita.
# Usiamo 'base_transforms' e 'WasteDataset' esattamente come li hai definiti nella Cella 2!
train_eval_loader = DataLoader(WasteDataset(train_df, transform=base_transforms), batch_size=BATCH_SIZE, shuffle=False)

train_etichette_vere = []
train_etichette_predette = []

# Spegniamo i gradienti per risparmiare memoria
with torch.no_grad():
    for inputs, labels in train_eval_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        
        # Estraiamo la classe vincitrice
        _, predicted = torch.max(outputs.data, 1)
        
        train_etichette_vere.extend(labels.cpu().numpy())
        train_etichette_predette.extend(predicted.cpu().numpy())

# ==========================================
# 2. VALUTAZIONE SUL VALIDATION SET
# ==========================================
print("🧪 Analisi del Validation Set in corso...")
val_etichette_vere = []
val_etichette_predette = []
    
with torch.no_grad():
    for inputs, labels in val_loader: # Questo è il tuo val_loader originale
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        
        _, predicted = torch.max(outputs.data, 1)
            
        val_etichette_vere.extend(labels.cpu().numpy())
        val_etichette_predette.extend(predicted.cpu().numpy())

# ==========================================
# 3. CALCOLO METRICHE UFFICIALI (Prof)
# ==========================================
# Metriche per il Training Set
train_accuracy_bal = balanced_accuracy_score(train_etichette_vere, train_etichette_predette)
train_matrice_conf = confusion_matrix(train_etichette_vere, train_etichette_predette)

# Metriche per il Validation Set
val_accuracy_bal = balanced_accuracy_score(val_etichette_vere, val_etichette_predette)
val_matrice_conf = confusion_matrix(val_etichette_vere, val_etichette_predette)
    
print(f"\n✅ Esame completato!")
print(f"🔹 Immagini Train analizzate: {len(train_etichette_vere)}")
print(f"🔹 Immagini Validation analizzate: {len(val_etichette_vere)}")
print("-" * 40)
print(f"🏆 BALANCED ACCURACY TRAIN:      {train_accuracy_bal * 100:.2f}%") 
print(f"🏆 BALANCED ACCURACY VALIDATION: {val_accuracy_bal * 100:.2f}%") 
print("-" * 40)

# ==========================================
# 4. MATRICI DI CONFUSIONE VISUALI (HEATMAP)
# ==========================================
# Creiamo una figura larga per contenere due grafici affiancati
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Grafico 1: Mappa del Training (Sinistra, in Verde)
sns.heatmap(train_matrice_conf, annot=True, fmt='d', cmap='Greens', 
            xticklabels=class_names, yticklabels=class_names, ax=axes[0])
axes[0].set_title('Matrice di Confusione - TRAINING SET', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Predizione del Modello', fontsize=12)
axes[0].set_ylabel('Classe Vera', fontsize=12)
axes[0].tick_params(axis='x', rotation=45)

# Grafico 2: Mappa del Validation (Destra, in Blu)
sns.heatmap(val_matrice_conf, annot=True, fmt='d', cmap='Blues', 
            xticklabels=class_names, yticklabels=class_names, ax=axes[1])
axes[1].set_title('Matrice di Confusione - VALIDATION SET', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Predizione del Modello', fontsize=12)
axes[1].set_ylabel('Classe Vera', fontsize=12)
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

# ==========================================
# 5. ANALISI DEGLI ERRORI (Sul Validation)
# ==========================================
print("\n🔍 Ricerca delle immagini classificate in modo errato nel Validation Set...")

# Troviamo gli indici dove la predizione è diversa dalla verità
indici_errori = [i for i in range(len(val_etichette_vere)) if val_etichette_vere[i] != val_etichette_predette[i]]

print(f"❌ Trovati {len(indici_errori)} errori totali sul Validation.")

if len(indici_errori) > 0:
    print("Mostro i primi 4 errori più clamorosi:\n")
    # Resettiamo l'indice del dataframe per assicurarci che corrisponda alla lista
    val_df_reset = val_df.reset_index(drop=True)
    
    # Mostriamo massimo 4 immagini
    fig, axes = plt.subplots(1, min(4, len(indici_errori)), figsize=(16, 4))
    if len(indici_errori) == 1: axes = [axes] # Gestione caso singolo errore
    
    for ax, idx in zip(axes, indici_errori[:4]):
        img_path = val_df_reset.iloc[idx]['filepath']
        vera_label = class_names[val_etichette_vere[idx]]
        pred_label = class_names[val_etichette_predette[idx]]
        
        # Carichiamo e mostriamo l'immagine
        img = Image.open(img_path)
        ax.imshow(img)
        ax.axis('off')
        ax.set_title(f"Vera: {vera_label}\nPredetta: {pred_label}", 
                     color='red' if vera_label != pred_label else 'green',
                     fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()

# OPZIONALE: Memoria (Misurata solo per le GPU compatibili)
if torch.cuda.is_available():
    vram_usata = torch.cuda.max_memory_allocated() / (1024 * 1024)
    print(f"\n💻 VRAM massima usata in fase di Test: {vram_usata:.2f} MB")

NameError: name 'model' is not defined